# Visualizzazione dello Spazio Embedding

Projects the **combined_embeddings** (7604 × 768) to 2D using **UMAP** and visualises them
with **Plotly**: interactive chart with hover, click and zoom.

What this notebook shows:
- The semantic structure of the catalogue: which products the model considers similar
- Natural clusters by category (Skincare, Supplements, Hair Care, etc.)
- Where a query lands relative to the clusters

> **Runtime**: UMAP takes ~1-2 min on CPU. Results are saved to `data/`
> so subsequent runs are instant.


## Setup

In [1]:
import sys, warnings, os
warnings.filterwarnings("ignore")

sys.path.insert(0, "./src")

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

DATA = "../Mean-Squared-Terrors/data"

print("Loading data...")
index_df     = pd.read_csv(f"{DATA}/embedding_index_enriched.csv")
combined_emb = np.load(f"{DATA}/combined_embeddings.npy").astype(np.float32)
faiss_index  = faiss.read_index(f"{DATA}/faiss_index.bin")

print("Loading MPNet model...")
model = SentenceTransformer("all-mpnet-base-v2")

print(f"\nProducts: {len(index_df):,} | Embeddings: {combined_emb.shape}")
print("Setup complete.")


Loading data...
Loading MPNet model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7241.35it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Products: 7,554 | Embeddings: (7554, 768)
Setup complete.


## Infer Categories from Titles

In [2]:
from mean_squared_terrors.embedding_viz import infer_category

index_df["category"] = index_df["product_title"].apply(infer_category)

print("Category distribution:")
cat_counts = index_df["category"].value_counts()
for cat, n in cat_counts.items():
    bar = "█" * (n // 30)
    print(f"  {cat:<15} {n:>5} {bar}")


Category distribution:
  Other            4869 ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Supplements       840 ████████████████████████████
  Pain Relief       435 ██████████████
  Oral Care         417 █████████████
  Skin Care         380 ████████████
  Baby Care         256 ████████
  Men's Grooming    167 █████
  Eye Care           96 ███
  Hair Care          69 ██
  Women's Health     25 


## UMAP: Dimensionality Reduction to 2D

The first run takes ~1-2 minutes. Afterwards, results are saved to
`data/umap_coords_2d.npy` and `data/umap_model.pkl` for instant reload.


In [3]:
import pickle
from mean_squared_terrors.embedding_viz import compute_umap

COORDS_PATH = f"{DATA}/umap_coords_2d.npy"
MODEL_PATH  = f"{DATA}/umap_model.pkl"

if os.path.exists(COORDS_PATH) and os.path.exists(MODEL_PATH):
    print("Loading pre-computed UMAP from disk...")
    coords_2d  = np.load(COORDS_PATH)
    with open(MODEL_PATH, "rb") as f:
        umap_model = pickle.load(f)
    print(f"  Loaded: {coords_2d.shape}")
else:
    print("Computing UMAP (first run — ~1-2 min)...")
    coords_2d, umap_model = compute_umap(combined_emb, n_neighbors=20, min_dist=0.1)
    np.save(COORDS_PATH, coords_2d)
    with open(MODEL_PATH, "wb") as f:
        pickle.dump(umap_model, f)
    print(f"  Saved to {COORDS_PATH}")

print(f"\n2D coordinates: {coords_2d.shape}")


Loading pre-computed UMAP from disk...
Sun May  3 13:11:54 2026 Building and compiling search function
  Loaded: (7554, 2)

2D coordinates: (7554, 2)


## Visualisation by Category

In [8]:
from mean_squared_terrors.embedding_viz import plot_embedding_space, save_html

fig = plot_embedding_space(
    coords_2d, index_df,
    color_by="category",
    title="DiscoverAI: Semantic Space · Health & Personal Care",
)
fig.show()

# Also save as standalone HTML (opens in browser, great for presentation)
save_html(fig, f"{DATA}/embedding_space_category.html")
print("\nClick categories in the legend to hide/show them.")
print("Zoom with scroll, pan with click+drag, hover for product details.")


Chart saved to: ../Mean-Squared-Terrors/data/embedding_space_category.html

Click categories in the legend to hide/show them.
Zoom with scroll, pan with click+drag, hover for product details.


## Alternative Colouring: Price Bucket

In [9]:
fig_price = plot_embedding_space(
    coords_2d, index_df,
    color_by="price_bucket",
    title="DiscoverAI: Semantic Space · Price Bucket",
)
fig_price.show()
save_html(fig_price, f"{DATA}/embedding_space_price.html")


Chart saved to: ../Mean-Squared-Terrors/data/embedding_space_price.html


## Projecting a Query into the Space

In [6]:
from mean_squared_terrors.embedding_viz import plot_query_in_space

# Try different queries — observe where they land relative to the clusters
QUERY = "affordable vitamin C serum anti-aging"

fig_query = plot_query_in_space(
    query=QUERY,
    model=model,
    umap_model=umap_model,
    coords_2d=coords_2d,
    index_df=index_df,
    faiss_index=faiss_index,
    top_k=5,
    color_by="category",
)
fig_query.show()
save_html(fig_query, f"{DATA}/embedding_query_example.html")
print(f"\nThe yellow star shows where '{QUERY}' falls in the semantic space.")
print("The red circles are the 5 nearest products — the same ones search_v3 would return.")


Epochs completed: 100%| ██████████ 100/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


Chart saved to: ../Mean-Squared-Terrors/data/embedding_query_example.html

The yellow star shows where 'affordable vitamin C serum anti-aging' falls in the semantic space.
The red circles are the 5 nearest products — the same ones search_v3 would return.


## Comparing Multiple Queries: Where Do They Land?

In [7]:
import plotly.graph_objects as go
from mean_squared_terrors.embedding_viz import _CATEGORY_COLORS, plot_embedding_space

# Show multiple queries together on the same chart
test_queries = [
    ("moisturizer for dry skin",         "#FF4136"),
    ("vitamin D supplement",             "#2ECC71"),
    ("shampoo for hair loss",            "#FF851B"),
    ("joint pain relief glucosamine",    "#B10DC9"),
]

# Base: all products in light grey
fig_multi = go.Figure()

# Product dots in grey
fig_multi.add_trace(go.Scatter(
    x=coords_2d[:,0], y=coords_2d[:,1],
    mode="markers",
    marker=dict(size=3, color="#E0E0E0", opacity=0.5),
    name="Products",
    hoverinfo="skip",
))

# One star per query
for query, color in test_queries:
    q_vec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    q_2d  = umap_model.transform(q_vec)
    fig_multi.add_trace(go.Scatter(
        x=[q_2d[0,0]], y=[q_2d[0,1]],
        mode="markers+text",
        marker=dict(symbol="star", size=16, color=color,
                    line=dict(color="#000", width=1)),
        text=[f"  {query[:30]}"],
        textposition="middle right",
        textfont=dict(size=11),
        name=query[:40],
        hovertemplate=f"<b>{query}</b><extra></extra>",
    ))

fig_multi.update_layout(
    title="Position of different queries in the semantic space",
    xaxis=dict(title="UMAP dim 1", showgrid=False, zeroline=False),
    yaxis=dict(title="UMAP dim 2", showgrid=False, zeroline=False),
    plot_bgcolor="white", paper_bgcolor="white",
    width=1000, height=650,
    legend=dict(bgcolor="rgba(255,255,255,0.9)", bordercolor="#ccc", borderwidth=1),
)
fig_multi.show()
save_html(fig_multi, f"{DATA}/embedding_multi_query.html")
print("\nThis visualisation shows how semantically different queries")
print("fall in different regions of the space — a visual validation of the model.")


Epochs completed: 100%| ██████████ 100/100 [00:00]


	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


Epochs completed: 100%| ██████████ 100/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs



Epochs completed: 100%| ██████████ 100/100 [00:00]


	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


Epochs completed: 100%| ██████████ 100/100 [00:00]

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


Chart saved to: ../Mean-Squared-Terrors/data/embedding_multi_query.html

This visualisation shows how semantically different queries
fall in different regions of the space — a visual validation of the model.


## Final Notes

The generated HTML files (`data/embedding_*.html`) open directly in the browser
and are fully standalone, ideal for the final presentation without needing Jupyter.

**What to look for:**
- Coherent clusters = the model has learned valid semantic representations
- Overlaps between categories = products spanning multiple domains (e.g. baby sunscreen)
- Queries landing in the right cluster = the retrieval is semantically correct
